In [1]:
import pandas as pd
import numpy as np
from scripts.db_connector import get_engine

In [ ]:
from scripts.processor import get_organic_data, get_ctr_data


target_id = 3
start = "2025-02-13"
end = "2025-12-29"
account_id = 3
date_start = "2025-02-13"
date_end = "2025-12-29"

In [3]:
get_organic_data(target_id, start, end).head()

,date_start,date_end,organic_impressions
0,2025-04-07,2025-04-13,36273
1,2025-04-14,2025-04-20,49912
2,2025-04-21,2025-04-27,61154
3,2025-04-28,2025-05-04,67339
4,2025-05-05,2025-05-11,56343


In [11]:
engine = get_engine()
    
# 1. 쿼리: apd와 ad를 JOIN하여 account_id 기준으로 데이터 추출

query = f"""
    SELECT apd.date, apd.impressions, apd.clicks
    FROM ad
    LEFT JOIN ad_performance_daily apd ON ad.ad_id = apd.ad_id
    WHERE account_id_legacy = '{account_id}'
        AND ad.created_time >= '{date_start}'
        -- date_end가 무슨 요일이든, 그 주의 월요일에서 하루를 뺀 '일요일'까지 조회
        AND ad.created_time <= (DATE_TRUNC('week', '{date_end}'::date) - INTERVAL '1 day')::date
        AND apd.date >= '{date_start}'
        AND apd.date <= DATE_TRUNC('week', '{date_end}'::date)::date
    ORDER BY date
"""

df = pd.read_sql(query, engine)

df.head()

,date,impressions,clicks
0,2025-02-13 09:00:00+00:00,70,1
1,2025-02-13 09:00:00+00:00,1,0
2,2025-02-13 09:00:00+00:00,50,0
3,2025-02-13 09:00:00+00:00,69,0
4,2025-02-13 09:00:00+00:00,1,0


In [8]:
engine = get_engine()
    
query = f"""
    SELECT 
        apd.age, apd.gender,
        ROUND((SUM(apd.clicks)::numeric / NULLIF(SUM(apd.impressions), 0)::numeric) * 100, 2) as ctr
    FROM ad
    LEFT JOIN ad_performance_daily apd ON ad.ad_id = apd.ad_id
    WHERE ad.ad_id = 1298
        AND ad.created_time >= '{date_start}'
        -- date_end가 무슨 요일이든, 그 주의 월요일에서 하루를 뺀 '일요일'까지 조회
        AND ad.created_time <= (DATE_TRUNC('week', '{date_end}'::date) - INTERVAL '1 day')::date
        AND apd.date >= '{date_start}'
        AND apd.date <= DATE_TRUNC('week', '{date_end}'::date)::date
    GROUP BY apd.age, apd.gender
    ORDER BY ctr DESC;
"""

df = pd.read_sql(query, engine)

In [7]:
df.head()

,age,gender,ctr
0,55-64,unknown,20.00
1,35-44,unknown,11.11
2,45-54,unknown,7.69
3,65+,female,4.74
4,45-54,male,4.68


In [22]:
engine = get_engine()
    
query = f"""
    SELECT 
    apd.age, 
    apd.gender, 
    AVG(apd.impressions) AS impressions, 
    AVG(apd.clicks) AS clicks,
    -- NULLIF를 사용하여 분모(impressions)가 0이면 NULL로 처리
    -- CTR 공식은 (클릭 / 노출) * 100입니다.
    ROUND(
        (SUM(apd.clicks)::numeric / NULLIF(SUM(apd.impressions), 0)::numeric) * 100, 
        2
    ) AS ctr
    FROM ad
    LEFT JOIN ad_performance_daily apd ON ad.ad_id = apd.ad_id
    WHERE ad.account_id = '{account_id}'
        AND ad.created_time >= '{date_start}'
        AND ad.created_time <= (DATE_TRUNC('week', '{date_end}'::date) - INTERVAL '1 day')::date
        AND apd.date >= '{date_start}'
        AND apd.date <= (DATE_TRUNC('week', '{date_end}'::date) - INTERVAL '1 day')::date
    GROUP BY apd.age, apd.gender
"""

df = pd.read_sql(query, engine)

df.head()

,age,gender,impressions,clicks,ctr
0,18-24,female,72.478409,1.886523,2.60
1,18-24,male,91.857822,1.785941,1.94
2,18-24,unknown,3.228291,0.062325,1.93
3,25-34,female,74.474321,2.759813,3.71
4,25-34,male,128.847859,3.162653,2.45


In [4]:
engine = get_engine()
    
order_direction = "ASC"

# 2. 개별 광고 데이터 가져오기 (uploaded_at, ig_permalink 포함)

ads_query = f"""
SELECT 
    ad.ad_id, 
    ad.ad_name,
    ad.ig_timestamp as uploaded_at, -- 업로드일로 사용
    ad.ig_permalink as thumbnail, -- 썸네일 경로로 사용
    ROUND((SUM(apd.clicks)::numeric / NULLIF(SUM(apd.impressions), 0)::numeric) * 100, 2) as ctr
FROM ad 
LEFT JOIN ad_performance_daily apd ON apd.ad_id = ad.ad_id
WHERE ad.account_id = '{account_id}'
    AND ad.created_time >= '{date_start}'
    -- date_end가 무슨 요일이든, 그 주의 월요일에서 하루를 뺀 '일요일'까지 조회
    AND ad.created_time <= (DATE_TRUNC('week', '{date_end}'::date) - INTERVAL '1 day')::date
    AND apd.date >= '{date_start}'
    AND apd.date <= DATE_TRUNC('week', '{date_end}'::date)::date
GROUP BY ad.ad_id
HAVING SUM(apd.impressions) >= 500
ORDER BY ctr {order_direction}
LIMIT 3;
"""

ads_df = pd.read_sql(ads_query, engine)

In [5]:
ads_df.head()

,ad_id,ad_name,uploaded_at,thumbnail,ctr
0,4141,"미래를 준비한다면, SNS를 시작하세요.",2025-06-29 09:00:23+00:00,https://www.instagram.com/p/DLeplA2JDz4/,0.11
1,4266,"매달, 성과 좋은 콘텐츠만 남깁니다",NaT,None,0.38
2,4284,고객의 리뷰 한 줄이 브랜드를 만듭니다,2025-05-06 09:00:35+00:00,https://www.instagram.com/p/DJTmsZZJUSl/,0.53


In [6]:
engine = get_engine()
    
    # 1. 쿼리: apd와 ad를 JOIN하여 account_id 기준으로 데이터 추출

query = f"""
    SELECT apd.date, apd.impressions, apd.clicks
    FROM ad
    LEFT JOIN ad_performance_daily apd ON ad.ad_id = apd.ad_id
    WHERE account_id = '{account_id}'
        AND ad.created_time >= '{date_start}'
        -- date_end가 무슨 요일이든, 그 주의 월요일에서 하루를 뺀 '일요일'까지 조회
        AND ad.created_time <= (DATE_TRUNC('week', '{date_end}'::date) - INTERVAL '1 day')::date
        AND apd.date >= '{date_start}'
        AND apd.date <= DATE_TRUNC('week', '{date_end}'::date)::date
"""


df = pd.read_sql(query, engine)

In [8]:
# date 최소값 최대값
if not df.empty:
    min_date = df['date'].min()
    max_date = df['date'].max()
    print(f"Date Range: {min_date} to {max_date}")

Date Range: 2025-02-13 09:00:00+00:00 to 2025-12-29 09:00:00+00:00


In [9]:
df.tail()

,date,impressions,clicks
67898,2025-12-29 09:00:00+00:00,9,0
67899,2025-12-29 09:00:00+00:00,1,0
67900,2025-12-29 09:00:00+00:00,26,1
67901,2025-12-29 09:00:00+00:00,21,1
67902,2025-12-29 09:00:00+00:00,47,0


In [3]:
_,_,f = get_content_performance_data(target_id, start, end, is_top=True)

In [4]:
f.head()

,ad_id,ad_name,uploaded_at,thumbnail,ad_total_imp,ctr
0,4066,전문직은 인플루언서가 되면 유리합니다.,2025-07-26 11:00:44+00:00,https://www.instagram.com/reel/DMkYve7JE4k/,5004,7.49
1,4024,"25y8m3w-5 왜 맥주 광고는 남자를, 소주 광고는 여자를 모델로 쓸까",2025-08-12 09:00:56+00:00,https://www.instagram.com/p/DNP8mL5PTSe/,6721,6.84
2,4170,코스메틱 브랜드 운영하시는 대표님!,2025-06-19 13:00:14+00:00,https://www.instagram.com/p/DLFVEdIJILI/,1560,6.79


In [ ]:
query = f"""
        SELECT 
            MIN(ig_timestamp) AS start_date,
            -- 가장 늦은 날짜의 직후 월요일 계산
            MAX(ig_timestamp) AS end_date
        FROM ad
        WHERE account_id = '{account_id}'
            AND created_time >= '{date_start}'
            -- date_end가 무슨 요일이든, 그 주의 월요일에서 하루를 뺀 '일요일'까지 조회
            AND created_time <= (DATE_TRUNC('week', '{date_end}'::date) - INTERVAL '1 day')::date
    """

In [ ]:
def get_bottom_ads_and_target_analysis(account_id, date_start, date_end):
    engine = get_engine()
    
    # 1. 0.05% 필터를 통과한 '하위' 3개 ad_id 가져오기 (ORDER BY CTR ASC)
    bottom_ads_query = f"""
    WITH total_stats AS (
        SELECT SUM(impressions) as total_site_imp
        FROM ad_performance_daily apd
        JOIN ad ON apd.ad_id = ad.ad_id
        WHERE ad.account_id = '{account_id}'
          AND apd.date BETWEEN '{date_start}' AND '{date_end}'
          AND apd.impressions > 0
    )
    SELECT 
        apd.ad_id, 
        ad.ad_name,
        SUM(apd.impressions) as ad_total_imp,
        ROUND((SUM(apd.clicks)::numeric / NULLIF(SUM(apd.impressions), 0)::numeric) * 100, 2) as ad_total_ctr
    FROM ad_performance_daily apd
    JOIN ad ON apd.ad_id = ad.ad_id, total_stats
    WHERE ad.account_id = '{account_id}'
      AND apd.date BETWEEN '{date_start}' AND '{date_end}'
      AND apd.impressions > 0
    GROUP BY apd.ad_id, ad.ad_name, total_stats.total_site_imp
    HAVING SUM(apd.impressions) >= total_stats.total_site_imp * 0.0005
    ORDER BY ad_total_ctr ASC
    LIMIT 3;
    """
    bottom_ads_df = pd.read_sql(bottom_ads_query, engine)
    bottom_ad_ids = bottom_ads_df['ad_id'].tolist()

    if not bottom_ad_ids:
        return None, None

    # 2. 선정된 하위 3개 광고의 타겟 데이터 가져오기
    target_query = f"""
    SELECT 
        ad_id, gender, age,
        SUM(impressions) as impressions,
        ROUND((SUM(clicks)::numeric / NULLIF(SUM(impressions), 0)::numeric) * 100, 2) as ctr
    FROM ad_performance_daily
    WHERE ad_id IN ({','.join(map(str, bottom_ad_ids))})
      AND date BETWEEN '{date_start}' AND '{date_end}'
      AND impressions > 0
    GROUP BY ad_id, gender, age
    ORDER BY ad_id, ctr DESC; -- 타겟 효율도 높은 순으로 정렬
    """
    target_df = pd.read_sql(target_query, engine)
    
    return bottom_ads_df, target_df

In [ ]:
# 키워드별 CTR 상위 N개 가져오기
def get_top_ctr_keywords_direct(account_id, date_start, date_end, top_n=10):
    engine = get_engine()
    
    query = f"""
    WITH ad_performance AS (
        -- 1. 광고별 총 노출과 클릭을 먼저 요약 (중복 계산 방지)
        SELECT 
            apd.ad_id, 
            SUM(impressions) as total_imp, 
            SUM(clicks) as total_click
        FROM ad_performance_daily apd
        JOIN ad ON apd.ad_id = ad.ad_id
        WHERE ad.account_id = '{account_id}'
          AND apd.date BETWEEN '{date_start}' AND '{date_end}'
          AND apd.impressions > 0
        GROUP BY apd.ad_id
    ),
    expanded_keywords AS (
        -- 2. 키워드 배열을 낱개(Row)로 펼침
        SELECT 
            ad_id,
            unnest(essential_keywords || variable_keywords) as keyword
        FROM ad_keyword
    )
    -- 3. 키워드별 합산 및 CTR 계산
    SELECT 
        ek.keyword,
        SUM(ap.total_imp) as impressions,
        SUM(ap.total_click) as clicks,
        CASE 
            WHEN SUM(ap.total_imp) > 0 
            THEN ROUND((SUM(ap.total_click)::numeric / SUM(ap.total_imp)::numeric) * 100, 2)
            ELSE 0 
        END as ctr
    FROM expanded_keywords ek
    JOIN ad_performance ap ON ek.ad_id = ap.ad_id
    WHERE ek.keyword IS NOT NULL AND ek.keyword != ''
    GROUP BY ek.keyword
    HAVING SUM(ap.total_imp) >= 10  -- 신뢰도를 위해 노출 10회 이상만
    ORDER BY ctr DESC
    LIMIT {top_n};
    """
    return pd.read_sql(query, engine)